# 05 - Error curves and the detection limit

Notebook 04 asked *does the tracker work*. This one asks *when does it stop
working*, which is the actual research question.

The method: impose a displacement we chose, at a noise level we chose, measure
what comes back, and repeat across a grid of both. Every point has a known
answer, so every error is measurable.

The output is one number per noise level - the smallest ground movement that can
still be told apart from measurement noise. That number is the project's result.

1 pixel = 10 m on the ground. Chamoli crept roughly 0.2 px (2 m) per year.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.testbed import Testbed
from src.tracker import track_grid

DATA = "../data/chamoli"
settings = pd.read_csv(f"{DATA}/settings.csv", header=None, index_col=0).squeeze("columns")

raw = np.load(f"{DATA}/{settings['base_date']}.npy").astype("float64")
base = (raw - raw.min()) / (raw.max() - raw.min())

SCAR_ROW = int(settings["scar_row"])
SCAR_COL = int(settings["scar_col"])
SIDE = 50
half = SIDE // 2
BOX = (SCAR_ROW - half, SCAR_ROW + half, SCAR_COL - half, SCAR_COL + half)

WINDOW, STEP, MARGIN = 32, 8, 40
# Which reading the curves are built from. "centre" keeps every window whose
# centre lies in the block (49 of them); "interior" keeps only windows lying
# wholly inside (9). Fewer windows means a noisier median - see the check below.
READING = "centre"
METRES_PER_PX = 10

# Work on a crop around the block. The sweep below is 400 tracker runs; on the
# full 504x505 image that is 16 minutes, on a 130x130 crop it is about a minute.
# Nothing is lost: every window we read sits inside the block or just outside it.
CROP = (BOX[0] - MARGIN, BOX[1] + MARGIN, BOX[2] - MARGIN, BOX[3] + MARGIN)

testbed = Testbed(base, BOX)


def cut(img):
    return img[CROP[0]:CROP[1], CROP[2]:CROP[3]]


def block_windows(result, pad=0):
    """Grid cells inside the block. pad=0 keeps any window whose centre is
    inside; pad=WINDOW//2 keeps only windows lying wholly inside."""
    lo, hi = MARGIN + pad, MARGIN + SIDE - pad
    return np.ix_((result["rows"] >= lo) & (result["rows"] < hi),
                  (result["cols"] >= lo) & (result["cols"] < hi))


print(f"block {SIDE}x{SIDE} px, crop {CROP[1]-CROP[0]}x{CROP[3]-CROP[2]} px")

## One measurement

`Testbed.pair` builds the pair: the block moves, the sensor adds grain, and
(here) no cloud. One seed drives both, so the same seed always gives the same
pair - the sweep is reproducible.

Two readings come out of each run. `centre` keeps every window whose centre
falls in the block, including ones hanging over the edge. `interior` keeps only
windows lying wholly inside.

The curves below use `centre`. It takes in a handful of windows that straddle
the block's edge, which pulls the reading down slightly - but it is a median
over 49 windows instead of 9, and the 9 interior ones overlap so heavily that
they are nearly one measurement repeated. The cost in bias is a few percent;
the gain in scatter is a factor of six. The cell after the ratio table measures
that trade rather than assuming it.

In [ ]:
def measure(dx, noise, seed):
    before, after, _, _ = testbed.pair(dx=dx, dy=0.0, noise=noise, cloud=0.0, seed=seed)
    r = track_grid(cut(before), cut(after), window=WINDOW, step=STEP)
    return (float(np.nanmedian(r["dx"][block_windows(r, 0)])),
            float(np.nanmedian(r["dx"][block_windows(r, WINDOW // 2)])))


centre, interior = measure(dx=2.0, noise=0.1, seed=0)
print(f"imposed 2.000 -> centre {centre:.3f}, interior {interior:.3f}")

## The sweep

Ten displacements from 0 to 2 px, five noise levels, eight seeds each. Zero
displacement is included deliberately: it measures how much the tracker reports
when nothing moved at all, and that scatter is what sets the detection limit.

Noise is expressed as a fraction of the image's own contrast, so 0.1 means the
grain is a tenth as strong as the real variation in the scene.

About a minute.

In [ ]:
SHIFTS = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0]
NOISES = [0.0, 0.05, 0.1, 0.2, 0.4]
NSEED = 8

records = []
for noise in NOISES:
    for dx in SHIFTS:
        for seed in range(NSEED):
            c, i = measure(dx, noise, seed)
            records.append({"noise": noise, "true": dx, "seed": seed,
                            "centre": c, "interior": i})
    print(f"noise {noise:.2f} done")

runs = pd.DataFrame(records)
summary = (runs.groupby(["noise", "true"])[READING]
                .agg(["mean", "std"]).reset_index())

print(f"\n{len(runs)} runs\n")
print(summary.pivot(index="true", columns="noise", values="mean").round(3))

## What the tracker actually returns

Every column sits below the imposed value, and the gap is proportionally worst
at the small end - the end this project cares about.

In [ ]:
ratio = summary[summary["true"] > 0].copy()
ratio["ratio"] = ratio["mean"] / ratio["true"]
print(ratio.pivot(index="true", columns="noise", values="ratio").round(3))

In [ ]:
# Is the centre reading paying much for those edge windows? Compare the two
# at zero noise, where any difference can only be geometry.
clean = runs[(runs["noise"] == 0) & (runs["seed"] == 0) & (runs["true"] > 0)]
side = clean[["true", "centre", "interior"]].copy()
side["dilution"] = side["interior"] - side["centre"]
print(side.to_string(index=False))

spread = runs[runs["true"] == 0].groupby("noise")[["centre", "interior"]].std()
print("\nscatter at zero displacement (std across seeds):")
print(spread.round(4))


Read the top-left corner: a 0.1 px movement comes back as 0.07, even with no
noise at all. The tracker is not adding an error, it is **shrinking the answer**,
and it shrinks small movements hardest.

Chamoli crept about 0.2 px a year. At that size the tracker returns roughly 70%
of the truth before noise is even considered. Any detection limit worked out
below inherits that.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, hi in zip(axes, [2.1, 0.32]):
    for noise in NOISES:
        sub = summary[summary["noise"] == noise].sort_values("true")
        keep = sub["true"] <= hi
        ax.errorbar(sub["true"][keep], sub["mean"][keep], yerr=sub["std"][keep],
                    marker="o", ms=4, capsize=3, lw=1.2, label=f"noise {noise:.2f}")
    ax.plot([0, hi], [0, hi], "k--", lw=1, label="perfect")
    ax.set_xlabel("true displacement (px)")
    ax.set_ylabel("measured displacement (px)")
    ax.set_xlim(0, hi)

axes[0].set_title("full range")
axes[1].set_title("the range this project lives in")
axes[1].axvline(0.2, color="tab:red", lw=1, ls=":")
axes[1].text(0.205, 0.01, "Chamoli: 0.2 px/yr", fontsize=9, color="tab:red")
axes[0].legend(fontsize=9)
plt.tight_layout()
plt.show()

### Reading it

- **Every line sits under the dashed one.** That is the shrinkage, and it does
  not go away at low noise - the noise-free line is below the diagonal too. So
  the cause is in the method itself, not in the noise.
- **The error bars grow with noise.** At noise 0.4 a single measurement can land
  a long way from the line. That spread, not the shrinkage, is what sets the
  limit.
- **The bars at zero displacement are the important ones.** They show the
  tracker reporting movement where there was none. Anything smaller than that
  scatter is indistinguishable from nothing happening.

## The detection limit

A movement counts as detected when the reading it produces is larger than the
noise the tracker generates on its own. Taking three standard deviations of the
zero-displacement scatter as the bar:

> **the detection limit is the smallest true movement whose average reading
> clears 3x the zero-movement scatter.**

Three sigma is a convention, not a law. `K_SIGMA` below is the one knob - change
it and every number changes with it, which is worth saying out loud rather than
hiding.

In [ ]:
K_SIGMA = 3.0


def detection_limit(sub, k=K_SIGMA):
    """Smallest true displacement whose mean reading clears k x the noise floor.

    Linear interpolation between the two bracketing points, so the answer is not
    rounded up to whichever displacement happened to be on the grid.
    """
    sub = sub.sort_values("true")
    floor = k * float(sub.loc[sub["true"] == 0, "std"].iloc[0])
    if floor <= 0:
        return np.nan, floor          # noise-free: nothing to clear
    t = sub["true"].to_numpy()
    m = sub["mean"].to_numpy()
    for i in range(1, len(t)):
        if m[i] >= floor:
            if m[i] == m[i - 1]:
                return t[i], floor
            frac = (floor - m[i - 1]) / (m[i] - m[i - 1])
            return t[i - 1] + frac * (t[i] - t[i - 1]), floor
    return np.nan, floor              # never clears it in the range tested


limits = []
for noise in NOISES:
    lim, floor = detection_limit(summary[summary["noise"] == noise])
    limits.append({"noise": noise,
                   "floor_px": round(floor, 4),
                   "limit_px": round(lim, 3) if np.isfinite(lim) else np.nan,
                   "limit_m": round(lim * METRES_PER_PX, 2) if np.isfinite(lim) else np.nan,
                   "years_to_see_chamoli": round(lim / 0.2, 1) if np.isfinite(lim) else np.nan})

limits = pd.DataFrame(limits)
print(limits.to_string(index=False))

In [ ]:
show = limits.dropna(subset=["limit_px"])

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(range(len(show)), show["limit_m"], color="tab:blue", width=0.6)
ax.set_xticks(range(len(show)))
ax.set_xticklabels([f"{n:.2f}" for n in show["noise"]])
ax.set_xlabel("image noise (fraction of scene contrast)")
ax.set_ylabel("smallest detectable movement (m)")
ax.set_title(f"detection limit, {K_SIGMA:.0f} sigma")

ax.axhline(2.0, color="tab:red", lw=1.2, ls="--")
ax.text(len(show) - 0.45, 2.1, "Chamoli: 2 m/yr", fontsize=9, color="tab:red", ha="right")

for b, v in zip(bars, show["limit_m"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.08, f"{v:.2f} m",
            ha="center", fontsize=9)

plt.tight_layout()
plt.show()

### What this says

Where a bar sits below the red line, one year of Chamoli-rate creep is visible
in a single image pair. Where it sits above, one year is not enough and the
movement has to be accumulated over several years before it can be separated
from noise - which is what `years_to_see_chamoli` in the table counts.

That is the sentence the whole project has been working towards, and it is
falsifiable: anyone can rerun this and get the same curve.
